In [ ]:
# === 03_train.ipynb - train FastSpeech2 ===
# Run AFTER 02_preprocess.ipynb (needs preprocessed_data/ + train.txt/val.txt).
#
# Kaggle settings: Accelerator = GPU T4 x2, Internet = On.
#
# On 2xT4 the full 40k-step run takes about 5 h.  Checkpoints are written every
# 2000 steps, so if the session dies just set RESUME_STEP below and re-run
# cell 5 - nothing else is lost.
LANG = "Telugu"  # English run complete -> Telugu is now the active language
SSTF = "/kaggle/working/sstf"
RESUME_STEP = 0      # 0 = from scratch, else the step of the checkpoint to resume
ALLOW_CPU = False    # True only if you accept ~27 h of CPU training

import os, sys
os.chdir(SSTF)
sys.path.insert(0, SSTF)

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
      "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  gpu", i, torch.cuda.get_device_name(i))

if torch.cuda.is_available():
    total = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count()))
    print("total GPU memory: %.1f GB" % (total / 1024 ** 3))
else:
    msg = ("NO GPU. Kaggle: Settings > Accelerator > GPU T4 x2, then Restart. "
           "Training 40k steps on CPU takes ~27 h and will be killed by the "
           "session limit before it finishes.")
    print("ERROR:", msg)
    if not ALLOW_CPU:
        raise SystemExit(msg)


In [ ]:
# 1) Patch the reference repo before the first step.  Both fixes are string
#    replacements, safe to re-run, and take a second.
import re, subprocess, sys
from pathlib import Path

repo = Path(f"{SSTF}/third_party/FastSpeech2_{LANG}")
assert (repo / "train.py").exists(), f"repo missing: {repo}"

from common import fs2cfg
fs2cfg.render(LANG)   # make sure fs2_config/*.yaml exist before the check below

# (a) "RuntimeError: Found dtype Double but expected Float" from backward().
#     pitch/energy are stored as float64 and utils/tools.py casts only pitch,
#     so the energy target stays double.  No reprocessing needed: cast at load.
tools = repo / "utils" / "tools.py"
src = tools.read_text(encoding="utf-8")
src = src.replace("energies = torch.from_numpy(energies).to(device)",
                  "energies = torch.from_numpy(energies).float().to(device)")
tools.write_text(src, encoding="utf-8")

ds = repo / "dataset.py"
src = ds.read_text(encoding="utf-8")
for name in ("mel", "pitch", "energy"):
    src = src.replace(f"{name} = np.load({name}_path)",
                      f"{name} = np.load({name}_path).astype(np.float32)")
ds.write_text(src, encoding="utf-8")

# (b) The HiFi-GAN and model checkpoints carry CUDA storage.  Without
#     map_location torch refuses to load them on a CPU-only kernel, and
#     get_vocoder runs before step 1.
model_py = repo / "utils" / "model.py"
src = model_py.read_text(encoding="utf-8")
src, n_map = re.subn(
    r"torch\.load\((?![^()]*map_location)([^()]*?),\s*weights_only=False\s*\)",
    lambda m: f"torch.load({m.group(1)}, map_location=device, weights_only=False)",
    src,
)
model_py.write_text(src, encoding="utf-8")
print("map_location added to", n_map, "torch.load() call(s)")

# Verify with a real batch from the real loader: no float64 may survive.
check = """
import torch, yaml
from dataset import Dataset
from utils.tools import to_device
pre = yaml.safe_load(open(r'__PRE__'))
tr = yaml.safe_load(open(r'__TRAIN__'))
ds = Dataset(pre['path']['preprocessed_path'] + '/train.txt', pre, tr)
batch = ds.collate_fn([ds[i] for i in range(min(4, len(ds)))])[0]
out = to_device(batch, torch.device('cpu'))
bad = [(i, str(t.dtype)) for i, t in enumerate(out)
       if torch.is_tensor(t) and t.is_floating_point() and t.dtype != torch.float32]
print('float64 tensors in a real batch:', bad)
assert not bad, bad
print('OK: every float tensor in the batch is float32')
"""
check = check.replace("__PRE__", f"{SSTF}/project/{LANG}/fs2_config/preprocess.yaml")
check = check.replace("__TRAIN__", f"{SSTF}/project/{LANG}/fs2_config/train.yaml")

r = subprocess.run([sys.executable, "-c", check], cwd=repo, capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-2000:])
assert r.returncode == 0, "dtype patch did not take effect"


In [ ]:
# 2) Audit the features.  Catches, in ~20 s, the data defects that would
#    otherwise kill a 5-hour run at the first bad batch:
#      * pitch/energy shorter than the phone sequence
#      * sum(duration) != mel frames
#    Both are repaired in place (pad / rescale) instead of re-running the 2 h
#    alignment.  A phone-count mismatch is fatal and reported.
import subprocess, sys, os

env = os.environ.copy()
env["PYTHONPATH"] = SSTF + (os.pathsep + env["PYTHONPATH"]) if "PYTHONPATH" in env else SSTF
r = subprocess.run([sys.executable, "-m", "common.dataset_check", "--lang", LANG],
                   cwd=SSTF, capture_output=True, text=True, env=env)
print(r.stdout[-6000:] or r.stderr[-3000:])
assert r.returncode == 0, "dataset audit failed - see the report above"


In [ ]:
# 3) Smoke test: 40 steps with logging every 10, validation + checkpoint every
#    20.  That exercises forward/backward, log.txt, evaluate() on val.txt,
#    HiFi-GAN synthesis, torch.save, and --restore-step - in about 2 minutes.
#    Then it wipes the smoke artefacts so the real curves start clean.
import glob, os, shutil, subprocess, sys
from pathlib import Path
import yaml

def reset_output():
    for d in ("ckpt", "log", "result"):
        shutil.rmtree(f"project/{LANG}/output/{d}", ignore_errors=True)

# Never delete a real run's checkpoints just because this cell was re-run.
existing = sorted(glob.glob(f"project/{LANG}/output/ckpt/*.pth.tar"))
if existing:
    last = max(int(os.path.basename(c).split(".")[0]) for c in existing)
    if last > 100:
        raise SystemExit(
            f"{len(existing)} checkpoint(s) up to step {last} already exist; "
            "skipping the smoke test so nothing is deleted. "
            "Run cell 5 to continue training.")

# Check if val data exists; if not, skip validation in smoke test
pre = yaml.safe_load(open(f"{SSTF}/project/{LANG}/fs2_config/preprocess.yaml"))
val_txt = Path(pre["path"]["preprocessed_path"]) / "val.txt"
has_val = False
if val_txt.exists():
    lines = [l for l in val_txt.read_text(encoding="utf-8").splitlines() if l.strip()]
    has_val = len(lines) > 0 and all(len(l.split("|")) == 4 for l in lines)
print(f"val.txt has valid data: {has_val}")

smoke = [sys.executable, f"project/{LANG}/train.py",
         "--steps", "40", "--log-every", "10"]
if has_val:
    smoke += ["--val-every", "20", "--save-every", "20"]
else:
    smoke += ["--val-every", "1000000", "--save-every", "20"]  # disable val

reset_output()
r = subprocess.run(smoke, cwd=SSTF)
assert r.returncode == 0, f"smoke run failed with {r.returncode}"

ckpts = sorted(glob.glob(f"project/{LANG}/output/ckpt/*.pth.tar"))
assert ckpts, "smoke run produced no checkpoint"
print("smoke checkpoints:", [os.path.basename(c) for c in ckpts])

# resume from the oldest smoke checkpoint: proves the 9 h-session recovery path
resume_from = int(os.path.basename(ckpts[0]).split(".")[0])
r = subprocess.run(smoke + ["--restore-step", str(resume_from)], cwd=SSTF)
assert r.returncode == 0, f"resume from step {resume_from} failed with {r.returncode}"
print(f"resume from step {resume_from}: OK")

reset_output()
print("smoke artefacts removed; ready for the full run")


In [ ]:
# 4) Full run: config train.max_steps = 40000 (~5 h on 2xT4).
#    If the session dies, read the last step from the log, put it in RESUME_STEP
#    in cell 1 and run this cell again.
import subprocess, sys

cmd = [sys.executable, f"project/{LANG}/train.py"]
if RESUME_STEP:
    cmd += ["--restore-step", str(RESUME_STEP)]
print("running:", " ".join(cmd))
r = subprocess.run(cmd, cwd=SSTF)
if r.returncode != 0:
    raise RuntimeError(f"train.py exited with {r.returncode} - see the output above")


In [ ]:
# 5) (Re)build the loss curves from output/log/*/log.txt.  train.py also does
#    this at the end of every successful run; this cell is the manual fallback.
import subprocess, sys
r = subprocess.run([sys.executable, f"project/{LANG}/train.py", "--plot-only"], cwd=SSTF)
assert r.returncode == 0


In [ ]:
# 6) Verify what the report and inference step need.
import glob, os, json
from pathlib import Path

ckpts = sorted(glob.glob(f"project/{LANG}/output/ckpt/*.pth.tar"))
print(f"{len(ckpts)} checkpoint(s):")
for c in ckpts:
    print("  %-60s %.0f MB" % (c, os.path.getsize(c) / 1e6))
assert ckpts, "no checkpoint - training did not finish"

for f in ("training_loss.png", "validation_loss.png"):
    p = Path(f"project/Results/{LANG}/{f}")
    print(f"{p}: {'%.0f kB' % (p.stat().st_size / 1e3) if p.exists() else 'MISSING'}")
    assert p.exists(), f"missing {p}"

for f in ("train", "val"):
    log = Path(f"project/{LANG}/output/log/{f}/log.txt")
    lines = log.read_text(encoding="utf-8").splitlines() if log.exists() else []
    print(f"{f} log: {len(lines)} entries | last: {lines[-1] if lines else '-'}")
    assert lines, f"empty {f} log"

print("\nDone. Project the loss curves are in project/Results/%s/" % LANG)


In [ ]:
# 7) Inference IN THIS SESSION.  A different notebook gets its own empty
#    /kaggle/working, so notebook 04 can never see these checkpoints - doing it
#    here means training and synthesis share one session and one download.
import subprocess, sys
r = subprocess.run([sys.executable, f"project/{LANG}/inference.py", "--step", "0"], cwd=SSTF)
assert r.returncode == 0, "inference failed - paste the traceback above"
print("inference done")


In [ ]:
# 8) Verify the assignment deliverables for this language exist.
import glob, json
from pathlib import Path

base = f"project/Results/{LANG}"
wavs = sorted(glob.glob(f"{base}/test_generated_audios/*.wav"))
durs = sorted(glob.glob(f"{base}/variance_adaptor_plots/duration/*.png"))
pit = sorted(glob.glob(f"{base}/variance_adaptor_plots/pitch/*.png"))
ene = sorted(glob.glob(f"{base}/variance_adaptor_plots/energy/*.png"))
csvs = sorted(glob.glob(f"{base}/variance_adaptor_plots/*.csv"))
print(f"wavs {len(wavs)}/5 | duration {len(durs)}/5 | pitch {len(pit)}/5 "
      f"| energy {len(ene)}/5 | csv {len(csvs)}/5")
assert len(wavs) >= 5 and len(durs) >= 5 and len(pit) >= 5 and len(ene) >= 5, \
    "missing deliverables - see the counts above"
for f in ("training_loss.png", "validation_loss.png"):
    assert Path(f"{base}/{f}").exists(), f"missing {base}/{f}"
print("all deliverables present for", LANG)


In [ ]:
# 9) Stage the downloads at the top of /kaggle/working so they appear in the
#    Output panel.  Download BOTH before closing or restarting the session -
#    /kaggle/working is deleted the moment the session ends.
import glob, shutil, zipfile
from pathlib import Path

ckpts = sorted(glob.glob(f"project/{LANG}/output/ckpt/*.pth.tar"),
               key=lambda p: int(Path(p).name.split(".")[0]))
assert ckpts, "no checkpoint - training did not finish"
final = Path(ckpts[-1])
staged_ckpt = Path(f"/kaggle/working/{LANG}_{final.name.split('.')[0]}.pth.tar")
shutil.copy2(final, staged_ckpt)

bundle = Path(f"/kaggle/working/{LANG}_bundle.zip")
extra = [
    f"project/{LANG}/config.yaml",
    f"project/{LANG}/data/vocab.json",
    f"project/{LANG}/data/train.txt",
    f"project/{LANG}/data/val.txt",
    f"project/{LANG}/data/split_stats.json",
    f"project/{LANG}/output/log/train/log.txt",
    f"project/{LANG}/output/log/val/log.txt",
    f"project/{LANG}/fs2_config/train.yaml",
    f"project/{LANG}/fs2_config/preprocess.yaml",
]
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(f"project/Results/{LANG}"):
        for f in files:
            p = Path(root) / f
            z.write(p, str(p))
    for f in extra:
        if Path(f).exists():
            z.write(f, f)

print("DOWNLOAD THESE TWO FILES (right sidebar -> Output -> refresh):")
for p in (bundle, staged_ckpt):
    print(f"  {p}   ({p.stat().st_size/1e6:.1f} MB)")
print("\nDo NOT restart or close this session until both are on your laptop.")
